In [1]:
%%capture
!pip install -U lightautoml

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
import torch
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.tasks import Task

In [5]:
N_THREADS = 4
N_FOLDS = 8
RANDOM_STATE = 42
TEST_SIZE = 0.2
TIMEOUT = 3600*100

numpy.random.seed(RANDOM_STATE)
torch.set_num_threads(N_THREADS)

train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
train = train.drop(columns=['id'])

train_x = train.drop(columns=['efficiency'])
train_y = train['efficiency']

In [8]:
task = Task('reg')

In [9]:
model = TabularAutoML(
    task = task,
    timeout = TIMEOUT,
    cpu_limit = N_THREADS,
    general_params = {'use_algos':[['cb', 'linear_l2', 'gbm', 'lgb', 'lgb_tuned']]},
    selection_params={'mode' : 0},
    tuning_params = {'max_tuning_time': 3600},
    reader_params = {'n_jobs': N_THREADS, 'cv': N_FOLDS, 'random_state': RANDOM_STATE}
)

out_of_fold_predictions = model.fit_predict(
    train,
    roles = {
        'target': 'efficiency',
        #'drop': ['unique_id']
        #'weights': 'weight'
    }, 
    verbose = 2
)

[09:14:41] Stdout logging level is INFO2.
[09:14:41] Task: reg

[09:14:41] Start automl preset with listed constraints:
[09:14:41] - time: 360000.00 seconds
[09:14:41] - CPU: 4 cores
[09:14:41] - memory: 16 GB

[09:14:41] Train data shape: (20000, 16)

[09:14:52] Layer 1 train process start. Time left 359989.04 secs
[09:14:53] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[09:14:53] ===== Start working with fold 0 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:54] ===== Start working with fold 1 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:54] ===== Start working with fold 2 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:55] ===== Start working with fold 3 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:55] ===== Start working with fold 4 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:56] ===== Start working with fold 5 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:57] ===== Start working with fold 6 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
[09:14:57] ===== Start working with fold 7 for Lvl_0_Pipe_

Optimization Progress: 100%|██████████| 101/101 [04:07<00:00,  2.45s/it, best_trial=74, best_value=-0.0114]

[09:19:17] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[09:19:17] The set of hyperparameters {'feature_fraction': 0.8142125566185491, 'num_leaves': 19, 'bagging_fraction': 0.5951759658034064, 'min_sum_hessian_in_leaf': 0.0845665422591458, 'reg_alpha': 0.687431132996711, 'reg_lambda': 4.509644205453004}
 achieve -0.0114 mse
[09:19:17] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...
[09:19:17] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====


[09:19:18] ===== Start working with fold 1 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:18] ===== Start working with fold 2 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:19] ===== Start working with fold 3 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:20] ===== Start working with fold 4 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:20] ===== Start working with fold 5 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:21] ===== Start working with fold 6 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:21] ===== Start working with fold 7 for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM =====
[09:19:22] Fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM finished. score = -0.010675168687260199
[09:19:22] Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM fitting and predicting completed
[09:19:22] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[09:19:22] ===== Start working with fold 0 for Lvl_0_Pipe_1_Mod_2_CatBoost =====
[09:19:23] ===== Start working with fold 1 for Lvl_0_Pipe_1_Mod_2_CatBoost =====
[

In [10]:
#print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

In [11]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([[0.4079078 ],
       [0.53489685],
       [0.5185892 ],
       ...,
       [0.6006742 ],
       [0.43148255],
       [0.558457  ]], dtype=float32)

In [12]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions.data.reshape(-1),
})
submission

,id,efficiency
0,0,0.407908
1,1,0.534897
2,2,0.518589
3,3,0.463576
4,4,0.460334
...,...,...
11995,11995,0.539947
11996,11996,0.458275
11997,11997,0.600674
11998,11998,0.431483


In [13]:
submission.to_csv('submission.csv', index = False)